# 01a — Feeds: download, roles and analysis windows

| role | feed | country | file | analysed on |
|---|---|---|---|---|
| **domestic** | **ZSR_SK** — Železnice Slovenskej republiky (ŽSR), via Transitious (**snapshot of 2026-05-13**, the legacy run's file) | **SK** | `sk_zsr.gtfs.zip` | its **service span** |
| foreign | DRAHY_CZ — CZPTT / JrUtil, via Transitious (**snapshot of 2026-11-03**, whole year kept) | CZ | `cz_CZPTT_new.gtfs.zip` | its service overlap with ZSR_SK |
| foreign | MAV_HU — MÁV, via Transitious (**snapshot of 2026-05-19**) | HU | `hu_mav.gtfs.zip` | its service overlap with ZSR_SK |
| foreign | BAHN_AT — Mobilitätsverbünde Österreich, via Transitious (**snapshot of 2026-05-19**) | AT | `BAHN_AT.zip` | its service overlap with ZSR_SK |
| foreign | PL_IC — **PKP Intercity** (gtfs.kasznia.net, snapshot of 2026-05-18) | PL | `PL_kasznia.zip` | its service overlap with ZSR_SK |
| foreign | PL_REG — **all other Polish carriers** (PolishTrainsGTFS, snapshot of 2026-06-06; PKP Intercity left out) | PL | `polish-trains.zip` | its service overlap with ZSR_SK |
| foreign | UKRZ_UA — Ukrzaliznytsya (OSM), via Transitious (**snapshot of 2026-07-09**) | UA | `ua-ukrzaliznytsya.gtfs.zip` | its service overlap with ZSR_SK |

**The domestic feed is ZSR_SK** (`cfg.domestic_feed`). Košice is reached through it; the five foreign
feeds are each analysed **separately** on their own overlap with it. The table is regenerated below
from the config and written to `output/kosice/feed_roles.csv`.

First step of the split pipeline (`plan_part_a_tagging_refactor.md`): **01a feeds → 01b stations →
02 domestic / 03 foreign**. Code: `traintranv2/nb01a_feeds.py`. Legacy `ksc_analysis.ipynb` and
`output/ke/` are untouched.

| writes | content |
|---|---|
| `data/feeds/feeds.manifest.json` | file, size, sha256 and calendar span of every feed (REP-2) |
| `output/kosice/feed_roles.csv` (+ `.meta.md`) | role, file, provider, window of every feed |
| `output/kosice/windows.json` | analysis window per feed (read by 01b, 02, 03) |

In [1]:
import os, sys

# Run from the project root so every relative path ('data/...', 'traintranv2/...') resolves the
# same whether the notebook is launched from notebooks/ or from the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import json
from pathlib import Path

import pandas as pd

import traintranv2  # noqa: F401  (fixes PROJ on import)
from traintranv2.config import default_config
from traintranv2.nb01a_feeds import download_feeds, feed_manifest, feed_roles, pair_windows

cfg = default_config()
cfg.city_slug = 'kosice'                       # outputs -> output/kosice/ (legacy run: output/ke/)
OUT = Path(cfg.out_dir)
OUT.mkdir(parents=True, exist_ok=True)
FEEDS = list(cfg.feeds)
dom = cfg.feeds[cfg.domestic_feed]
print(f'DOMESTIC FEED : {cfg.domestic_feed} ({dom["country"]}, {dom["data_origin"]}, {dom["filename"]})')
print(f'FOREIGN FEEDS : {[k for k in FEEDS if k != cfg.domestic_feed]}')
print(f'output dir    : {OUT}')

DOMESTIC FEED : ZSR_SK (SK, Železnice Slovenskej Republiky (ŽSR), sk_zsr.gtfs.zip)
FOREIGN FEEDS : ['MAV_HU', 'UKRZ_UA', 'BAHN_AT', 'DRAHY_CZ', 'PL_IC', 'PL_REG']
output dir    : output\kosice


## 1. Download and manifest

Only missing files are downloaded (`[skip]` otherwise). **Every feed is an older snapshot** taken near the
start of the timetable year (SK, HU, AT in May 2026, UA in July) — except CZ, whose November file keeps the whole year, has 7–9 % more trains and carries UIC codes and parent stations: a snapshot published late in the year
keeps its calendar but drops the services already run (the current SK file of 2026-10-27 has ~3.7 instead of
~40 trains a day on the Košice – Žilina line before September; the current AT file none before September).
The current files are kept as `*_new` / under their Transitious names and not used. **Poland comes from two feeds**, because no full-year all-carrier Polish GTFS exists:
**PL_IC** = the PKP Intercity feed `PL_kasznia.zip` (2026-05-18 .. 08-26, 101 days), and **PL_REG** = the all-carrier
PolishTrainsGTFS `polish-trains.zip`, a rolling one-month feed (2026-06-06 .. 07-06), **with PKP Intercity left out**
(`excluded_agencies`, agency `IC`). So no train is in both feeds: the de-duplication is that agency filter. PL_REG
carries the regional lines toward the Slovak border (Muszyna – Plaveč, Zwardoń – Skalité, the seasonal Kraków –
Poprad trains). Each feed keeps its own window; how the two are combined is decided in notebooks 03.
Both Polish files split some through trains into legs joined by stay-seated transfers; notebooks 02 / 03 rebuild them
as one train (`nb02_connections.merge_stay_seated`). The domestic file is checked
by sha256: it must be the legacy run's (`9dc3c38b05fb…`).

In [3]:
download_feeds(cfg, keys=FEEDS)
manifest = pd.DataFrame(feed_manifest(cfg, keys=FEEDS))
display(manifest[['key', 'country', 'file', 'bytes', 'sha256', 'span_start', 'span_end', 'span_days']]
        .assign(MB=lambda x: (x.bytes / 1e6).round(1), sha256=lambda x: x.sha256.str[:12]).drop(columns='bytes'))

EXPECTED_SPANS = {'MAV_HU': ('2026-01-19', '2026-11-15'), 'UKRZ_UA': ('2025-12-14', '2026-12-12'),
                  'ZSR_SK': ('2025-12-14', '2026-12-12'), 'BAHN_AT': ('2025-12-14', '2026-12-12'),
                  'DRAHY_CZ': ('2025-12-14', '2026-12-12'), 'PL_IC': ('2026-05-18', '2026-08-26'),
                  'PL_REG': ('2026-06-04', '2026-07-06')}
assert manifest.set_index('key').loc[cfg.domestic_feed, 'sha256'].startswith('9dc3c38b05fb'),     'the domestic file is not the legacy snapshot (feed_version 2026-05-13)'
spans = manifest.set_index('key')[['span_start', 'span_end']]
for k, (a, b) in EXPECTED_SPANS.items():
    assert tuple(spans.loc[k]) == (a, b), f'{k}: span {tuple(spans.loc[k])} -- not the expected snapshot'
print('feed snapshots as expected; manifest ->', Path(cfg.feeds_dir) / 'feeds.manifest.json')

  [skip] ZSR_SK already at data/feeds\sk_zsr.gtfs.zip
  [skip] MAV_HU already at data/feeds\hu_mav.gtfs.zip
  [skip] UKRZ_UA already at data/feeds\ua-ukrzaliznytsya.gtfs.zip
  [skip] BAHN_AT already at data/feeds\BAHN_AT.zip
  [skip] DRAHY_CZ already at data/feeds\cz_CZPTT_new.gtfs.zip
  [skip] PL_IC already at data/feeds\PL_kasznia.zip
  [skip] PL_REG already at data/feeds\polish-trains.zip


,key,country,file,sha256,span_start,span_end,span_days,MB
0,ZSR_SK,SK,sk_zsr.gtfs.zip,9dc3c38b05fb,2025-12-14,2026-12-12,364,0.3
1,MAV_HU,HU,hu_mav.gtfs.zip,18aa41295c0d,2026-01-19,2026-11-15,301,2.3
2,UKRZ_UA,UA,ua-ukrzaliznytsya.gtfs.zip,0eecc9437741,2025-12-14,2026-12-12,364,3.5
3,BAHN_AT,AT,BAHN_AT.zip,58e7a3839d53,2025-12-14,2026-12-12,364,5.2
4,DRAHY_CZ,CZ,cz_CZPTT_new.gtfs.zip,9d4dbb07a60c,2025-12-14,2026-12-12,364,50.2
5,PL_IC,PL,PL_kasznia.zip,f835b03a78a1,2026-05-18,2026-08-26,101,109.6
6,PL_REG,PL,polish-trains.zip,a07734e3bf29,2026-06-04,2026-07-06,33,10.1


feed snapshots as expected; manifest -> data\feeds\feeds.manifest.json


## 1b. Coverage check: trains per day by month

Rail trains per day (bus routes and excluded agencies left out) from each feed's calendar, by month, over the
domestic calendar span. A feed whose months drop well below its usual level does not cover them, whatever its
calendar declares; the windows in §2 are cut to the days that carry service.

In [4]:
from traintranv2.nb01a_feeds import monthly_trains

coverage = monthly_trains(cfg, keys=FEEDS)
display(coverage)
coverage.to_csv(OUT / 'feed_monthly_trains.csv', encoding='utf-8')
print('[wrote]', OUT / 'feed_monthly_trains.csv')

,ZSR_SK,MAV_HU,UKRZ_UA,BAHN_AT,DRAHY_CZ,PL_IC,PL_REG
2025-12,1373,0,56,0,8050,0,0
2026-01,1523,1461,57,0,8647,0,0
2026-02,1800,3463,58,0,8742,0,0
2026-03,1807,3453,58,4075,8710,0,0
2026-04,1764,3481,59,6279,8689,0,0
2026-05,1820,3470,61,6049,8581,241,0
2026-06,1944,3520,64,6179,8965,535,4927
2026-07,1855,3518,71,6231,8862,549,1105
2026-08,1837,3480,71,6166,8798,462,0
2026-09,1959,3486,71,6458,8868,0,0


[wrote] output\kosice\feed_monthly_trains.csv


## 2. Roles and analysis windows

Windows come from the **days the trains actually run**, not from the calendar's declared span (§1b): a feed's
service span runs from the first to the last day with at least **50 %** of its median daily trains
(`cfg.window_min_share`). The **domestic** feed is analysed on its service span; every **foreign** feed on the
overlap of its service span with the domestic one, separately. `windows.json` keeps the declared calendar
span beside it (`calendar_start` / `calendar_end`).

In [5]:
from traintranv2.nb01a_feeds import write_meta

roles = feed_roles(cfg, keys=FEEDS)
display(roles)
assert roles.loc[roles.role == 'domestic', 'feed_key'].tolist() == [cfg.domestic_feed] == ['ZSR_SK']

windows = pair_windows(cfg, keys=FEEDS)
EXPECTED_DAYS = {'ZSR_SK': 364, 'BAHN_AT': 276, 'DRAHY_CZ': 364, 'PL_IC': 101, 'PL_REG': 32, 'UKRZ_UA': 364, 'MAV_HU': 301}
assert {k: v['days'] for k, v in windows.items()} == EXPECTED_DAYS, {k: v['days'] for k, v in windows.items()}
for k, v in windows.items():
    v['role'] = 'domestic' if k == cfg.domestic_feed else 'foreign'

roles.to_csv(OUT / 'feed_roles.csv', index=False, encoding='utf-8-sig')
(OUT / 'windows.json').write_text(json.dumps(windows, indent=2), encoding='utf-8')
print('[wrote]', OUT / 'feed_roles.csv', '|', OUT / 'windows.json')
print('[meta] ', write_meta(str(OUT / 'feed_roles.csv'), 'feed_roles', roles,
                            extra={'Domestic feed': cfg.domestic_feed}))

,feed_key,country,role,file,present,provider,data_origin,window_start,window_end,window_days,window_basis,snapshot
0,ZSR_SK,SK,domestic,sk_zsr.gtfs.zip,True,Transitious,Železnice Slovenskej Republiky (ŽSR),2025-12-14,2026-12-12,364,service span (days with >= 50% of the feed's m...,"Older snapshot, feed_version 2026-05-13 (sha25..."
1,MAV_HU,HU,foreign,hu_mav.gtfs.zip,True,Transitious,MÁV Személyszállítási Zrt.,2026-01-19,2026-11-15,301,service span (days with >= 50% of the feed's m...,"Older snapshot, valid 2026-01-19 .. 2026-11-15..."
2,UKRZ_UA,UA,foreign,ua-ukrzaliznytsya.gtfs.zip,True,Transitious,OpenStreetMap contributors,2025-12-14,2026-12-12,364,service span (days with >= 50% of the feed's m...,"Older snapshot, valid 2025-12-14 .. 2026-12-12..."
3,BAHN_AT,AT,foreign,BAHN_AT.zip,True,Transitious,Mobilitätsverbünde Österreich OG.,2026-03-12,2026-12-12,276,service span (days with >= 50% of the feed's m...,"Older snapshot, feed_version 2026-05-19: train..."
4,DRAHY_CZ,CZ,foreign,cz_CZPTT_new.gtfs.zip,True,JrUtil via Transitious,JrUtil,2025-12-14,2026-12-12,364,service span (days with >= 50% of the feed's m...,Snapshot of 2026-11-03 (sha256 9d4dbb07a60c......
5,PL_IC,PL,foreign,PL_kasznia.zip,True,"gtfs.kasznia.net (kasmar00), remix of the mkur...",PKP Intercity (via mkuran.pl / gtfs.kasznia.net),2026-05-18,2026-08-26,101,service span (days with >= 50% of the feed's m...,"PKP Intercity only (feed_version 2026-05-18, 1..."
6,PL_REG,PL,foreign,polish-trains.zip,True,"Mikolaj Kuranowski (PolishTrainsGTFS, schedule...","PKP PLK (Otwarte Dane Kolejowe), via PolishTra...",2026-06-05,2026-07-06,32,service span (days with >= 50% of the feed's m...,"All-carrier PolishTrainsGTFS, a rolling one-mo..."


[wrote] output\kosice\feed_roles.csv | output\kosice\windows.json
[meta]  output\kosice\feed_roles.csv.meta.md


## Next

`01b_stations_tag_match.ipynb` builds the station catalogs of all six feeds from these files.